# Detecção de Objetos e Reconhecimento de Gestos Personalizado

Este notebook utiliza o MediaPipe para detectar a mão e extrair os marcos (landmarks), e o **nosso modelo treinado (`modelo_gestos.pkl`)** para classificar o gesto em tempo real.

A visualização é feita com **OpenCV**.

In [23]:
import cv2
import mediapipe as mp
import numpy as np
import time
import os
import joblib

print(joblib)
print("Bibliotecas carregadas!")


<module 'joblib' from 'c:\\Users\\camil\\configgesture\\.venv\\Lib\\site-packages\\joblib\\__init__.py'>
Bibliotecas carregadas!


### Configuração dos Caminhos e Carregamento do Modelo

In [24]:
OBJECT_DETECTOR_MODEL = "../models/efficientdet_lite0.tflite" #detectordeobjeto
HAND_LANDMARKER_MODEL = "../models/hand_landmarker.task" #marcacaodelinhas
CUSTOM_GESTURE_MODEL = "../models/modelo_gestos.pkl" #modelodosgestos

for f in [OBJECT_DETECTOR_MODEL, HAND_LANDMARKER_MODEL, CUSTOM_GESTURE_MODEL]:
    if os.path.exists(f):
        print(f"✓ {f} encontrado.")
    else:
        print(f"✗ ERRO: {f} não encontrado!")

clf = joblib.load(CUSTOM_GESTURE_MODEL)
print("✓ Modelo customizado carregado com sucesso!")


✓ ../models/efficientdet_lite0.tflite encontrado.
✓ ../models/hand_landmarker.task encontrado.
✓ ../models/modelo_gestos.pkl encontrado.
✓ Modelo customizado carregado com sucesso!


### Funções de Utilitárias

In [25]:
def extract_features(hand_landmarks):
    """Converte landmarks do MediaPipe para o formato de features do modelo (x0, y0, z0, ...)."""
    features = []
    for lm in hand_landmarks:
        features.extend([lm.x, lm.y, lm.z])
    return np.array(features).reshape(1, -1)

def draw_hand_landmarks(frame, hand_landmarks):
    """Desenha os pontos e conexões da mão manualmente usando OpenCV."""
    HAND_CONNECTIONS = [
        (0, 1), (1, 2), (2, 3), (3, 4),
        (0, 5), (5, 6), (6, 7), (7, 8),
        (5, 9), (9, 10), (10, 11), (11, 12),
        (9, 13), (13, 14), (14, 15), (15, 16),
        (13, 17), (17, 18), (18, 19), (19, 20), (0, 17)
    ]
    h, w, _ = frame.shape
    for connection in HAND_CONNECTIONS:
        start_idx, end_idx = connection
        start_point = (int(hand_landmarks[start_idx].x * w), int(hand_landmarks[start_idx].y * h))
        end_point = (int(hand_landmarks[end_idx].x * w), int(hand_landmarks[end_idx].y * h))
        cv2.line(frame, start_point, end_point, (255, 255, 255), 2)
    for landmark in hand_landmarks:
        cx, cy = int(landmark.x * w), int(landmark.y * h)
        cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)


### Loop de Processamento (Webcam)
Pressione **'q'** para sair.

In [26]:
BaseOptions = mp.tasks.BaseOptions
ObjectDetector = mp.tasks.vision.ObjectDetector
ObjectDetectorOptions = mp.tasks.vision.ObjectDetectorOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options_objects = ObjectDetectorOptions(
    base_options=BaseOptions(model_asset_path=OBJECT_DETECTOR_MODEL),
    running_mode=VisionRunningMode.VIDEO,
    score_threshold=0.5
)

options_hands = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=HAND_LANDMARKER_MODEL),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2
)

cap = cv2.VideoCapture(0)
last_timestamp_ms = 0

with ObjectDetector.create_from_options(options_objects) as detector, \
     HandLandmarker.create_from_options(options_hands) as landmarker:

    print("Webcam pronta! Pressione 'q' na janela de vídeo para sair.")

    while cap.isOpened():
        success, frame = cap.read()
        if not success: break

        frame = cv2.flip(frame, 1)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)

        timestamp_ms = int(time.time() * 1000)
        if timestamp_ms <= last_timestamp_ms:
            timestamp_ms = last_timestamp_ms + 1
        last_timestamp_ms = timestamp_ms

        try:

            object_result = detector.detect_for_video(mp_image, timestamp_ms)
            for detection in object_result.detections:
                bbox = detection.bounding_box
                cv2.rectangle(frame, (bbox.origin_x, bbox.origin_y),
                              (bbox.origin_x + bbox.width, bbox.origin_y + bbox.height), (0, 255, 0), 2)
                label = f"{detection.categories[0].category_name}"
                cv2.putText(frame, label, (bbox.origin_x, bbox.origin_y - 10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

            hand_result = landmarker.detect_for_video(mp_image, timestamp_ms)

            if hand_result.hand_landmarks:
                for i, hand_landmarks in enumerate(hand_result.hand_landmarks):
                    draw_hand_landmarks(frame, hand_landmarks)

                    features = extract_features(hand_landmarks)
                    prediction = clf.predict(features)[0]
                    proba = np.max(clf.predict_proba(features))

                    label = f"{prediction} ({proba:.2f})"

                    px, py = int(hand_landmarks[0].x * frame.shape[1]), int(hand_landmarks[0].y * frame.shape[0])
                    cv2.putText(frame, label, (px, py + 30),
                                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)
        except Exception as e:

            pass

        cv2.imshow('Custom Gesture Recognition', frame)
        if cv2.waitKey(1) & 0xFF == ord('q'): break

cap.release()
cv2.destroyAllWindows()
print("Encerrado.")


Webcam pronta! Pressione 'q' na janela de vídeo para sair.
Encerrado.
